# Product Backorder Prediction -- Supply Chain Analytics

---

## Business Context and Problem Definition

In supply chain management, a backorder occurs when a customer places an order for a product that is temporarily out of stock. While backorders indicate demand, they also create delays, increase operational costs, erode customer trust, and can lead to lost revenue if customers cancel orders or switch to competitors.

The challenge for inventory managers is twofold: overstocking ties up capital and increases warehousing costs, while understocking leads to backorders. Predicting which products are likely to go on backorder allows companies to take proactive measures such as adjusting reorder points, expediting shipments from suppliers, or reallocating inventory across distribution centers.

This notebook builds an end-to-end machine learning pipeline that predicts whether a product will go on backorder using inventory levels, sales history, supplier performance metrics, and product risk flags. The dataset reflects a realistic supply chain scenario with class imbalance (most products do not go on backorder), missing values, and mixed feature types.

**What This Notebook Covers**

- Data ingestion and quality assessment
- Exploratory analysis of inventory and sales patterns
- Handling missing data and class imbalance with SMOTE
- Building sklearn Pipelines for reproducible preprocessing
- Training multiple classifiers including ensemble and boosting methods
- Hyperparameter tuning with cross-validation
- Evaluation using precision, recall, F1, ROC-AUC, and precision-recall curves
- Feature importance and business interpretation

## Library Setup

In [3]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_score, recall_score, f1_score, roc_auc_score,
                             roc_curve, precision_recall_curve, average_precision_score)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              AdaBoostClassifier, BaggingClassifier, VotingClassifier,
                              StackingClassifier, ExtraTreesClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

plt.rcParams['figure.dpi'] = 100
sns.set_palette('deep')
print("All libraries loaded successfully.")

All libraries loaded successfully.


## Data Ingestion

In [5]:
df = pd.read_csv('backorder_data.csv')

print("Dataset dimensions: {} rows, {} columns".format(df.shape[0], df.shape[1]))
print()
print("Column listing:")
for i, col in enumerate(df.columns):
    print("  {:2d}. {:25s} dtype: {}".format(i+1, col, df[col].dtype))

Dataset dimensions: 5000 rows, 23 columns

Column listing:
   1. sku                       dtype: int64
   2. national_inv              dtype: float64
   3. lead_time                 dtype: float64
   4. in_transit_qty            dtype: int64
   5. forecast_3_month          dtype: int64
   6. forecast_6_month          dtype: int64
   7. forecast_9_month          dtype: int64
   8. sales_1_month             dtype: int64
   9. sales_3_month             dtype: int64
  10. sales_6_month             dtype: int64
  11. sales_9_month             dtype: int64
  12. min_bank                  dtype: int64
  13. potential_issue           dtype: object
  14. pieces_past_due           dtype: int64
  15. perf_6_month_avg          dtype: float64
  16. perf_12_month_avg         dtype: float64
  17. local_bo_qty              dtype: int64
  18. deck_risk                 dtype: object
  19. oe_constraint             dtype: object
  20. ppap_risk                 dtype: object
  21. stop_auto_buy          

In [6]:
df.head(10)

,sku,national_inv,lead_time,in_transit_qty,forecast_3_month,forecast_6_month,forecast_9_month,sales_1_month,sales_3_month,sales_6_month,...,pieces_past_due,perf_6_month_avg,perf_12_month_avg,local_bo_qty,deck_risk,oe_constraint,ppap_risk,stop_auto_buy,rev_stop,went_on_backorder
0,10001,184.0,7.0,14,138,523,58,81,188,124,...,6,0.34,0.92,2,No,No,No,No,No,No
1,10002,1455.0,8.0,12,274,81,37,49,7,34,...,10,0.67,0.59,0,Yes,Yes,No,No,No,No
2,10003,608.0,21.0,5,165,170,556,267,189,47,...,3,0.64,0.92,1,No,No,No,No,No,Yes
3,10004,406.0,7.0,28,17,435,2030,19,18,69,...,1,0.82,0.75,0,Yes,No,No,No,No,No
4,10005,34.0,21.0,19,41,263,419,70,247,79,...,1,0.62,0.66,2,No,No,No,Yes,No,Yes
5,10006,34.0,2.0,60,49,536,31,96,98,97,...,5,0.85,1.00,0,Yes,No,Yes,No,No,No
6,10007,-21.0,14.0,0,59,1299,150,139,363,103,...,4,0.54,0.65,2,No,No,No,No,No,No
7,10008,955.0,21.0,30,166,49,168,41,571,212,...,3,0.50,0.50,3,No,No,No,No,No,Yes
8,10009,409.0,4.0,43,140,494,247,92,107,249,...,1,0.75,0.59,2,No,No,No,No,No,No
9,10010,NaN,7.0,42,45,104,26,28,123,129,...,10,0.77,0.87,1,No,No,No,No,No,No


In [7]:
df.describe()

,sku,national_inv,lead_time,in_transit_qty,forecast_3_month,forecast_6_month,forecast_9_month,sales_1_month,sales_3_month,sales_6_month,sales_9_month,min_bank,pieces_past_due,perf_6_month_avg,perf_12_month_avg,local_bo_qty
count,5000.000000,4873.000000,4850.000000,5000.000000,5000.00000,5000.000000,5000.000000,5000.000000,5000.00000,5000.000000,5000.000000,5000.000000,5000.000000,4843.000000,4854.000000,5000.000000
mean,12500.500000,445.205007,11.768247,29.881800,204.40200,411.586000,576.887200,79.583000,197.10460,352.050600,489.571000,49.785000,4.503200,0.740000,0.713780,2.518000
std,1443.520003,494.308846,10.921414,30.491097,205.00642,408.602614,580.392804,79.891532,195.52646,361.295254,487.546745,49.346365,4.941019,0.184665,0.187015,2.950833
min,10001.000000,-50.000000,2.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.070000,0.000000,0.000000
25%,11250.750000,90.000000,4.000000,8.000000,60.00000,119.000000,170.000000,22.000000,56.00000,99.000000,138.000000,14.000000,1.000000,0.620000,0.580000,0.000000
50%,12500.500000,297.000000,8.000000,20.000000,143.00000,288.000000,405.000000,55.000000,137.00000,236.000000,342.000000,35.000000,3.000000,0.750000,0.720000,2.000000
75%,13750.250000,639.000000,14.000000,42.000000,278.00000,566.250000,790.250000,110.000000,274.00000,490.000000,689.000000,69.000000,6.000000,0.890000,0.860000,4.000000
max,15000.000000,4036.000000,52.000000,256.000000,1899.00000,3688.000000,4934.000000,677.000000,1444.00000,3671.000000,5026.000000,437.000000,40.000000,1.000000,1.000000,26.000000


## Data Quality Assessment

In [9]:
# Missing value analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Missing Count': missing, 'Missing Percent': missing_pct})
missing_report = missing_report[missing_report['Missing Count'] > 0]

print("Columns with missing values:")
print(missing_report)
print()
print("Total cells with missing data: {}".format(df.isnull().sum().sum()))

Columns with missing values:
                   Missing Count  Missing Percent
national_inv                 127             2.54
lead_time                    150             3.00
perf_6_month_avg             157             3.14
perf_12_month_avg            146             2.92

Total cells with missing data: 580


In [10]:
# Duplicates check
print("Duplicate rows:", df.duplicated().sum())
print("Unique SKUs:", df['sku'].nunique(), "out of", len(df), "rows")

Duplicate rows: 0
Unique SKUs: 5000 out of 5000 rows


In [11]:
# Target distribution
target_counts = df['went_on_backorder'].value_counts()
print("Target variable distribution:")
print(target_counts)
print()
backorder_rate = target_counts['Yes'] / len(df) * 100
print("Backorder rate: {:.2f}%".format(backorder_rate))
print()
print("This confirms a class imbalance problem. The majority class (No) significantly")
print("outweighs the minority class (Yes), which will influence model selection and evaluation.")

Target variable distribution:
went_on_backorder
No     4298
Yes     702
Name: count, dtype: int64

Backorder rate: 14.04%

This confirms a class imbalance problem. The majority class (No) significantly
outweighs the minority class (Yes), which will influence model selection and evaluation.


## Exploratory Data Analysis

### Target and Inventory Distribution

In [14]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Target balance
colors_target = ['#3498db', '#e74c3c']
target_counts.plot(kind='bar', ax=axes[0], color=colors_target, edgecolor='black')
axes[0].set_title('Backorder Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Went on Backorder')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

# National inventory distribution
axes[1].hist(df['national_inv'].dropna(), bins=60, color='steelblue',
             edgecolor='black', alpha=0.7)
axes[1].set_title('National Inventory Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('National Inventory')
axes[1].set_ylabel('Frequency')
axes[1].axvline(df['national_inv'].median(), color='red', linestyle='--',
                label='Median: {:.0f}'.format(df['national_inv'].median()))
axes[1].legend()

# Lead time distribution
lt_counts = df['lead_time'].dropna().value_counts().sort_index()
axes[2].bar(lt_counts.index.astype(str), lt_counts.values, color='coral', edgecolor='black')
axes[2].set_title('Lead Time Distribution', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Lead Time (days)')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

### Sales and Forecast Patterns

In [16]:
sales_cols = ['sales_1_month', 'sales_3_month', 'sales_6_month', 'sales_9_month']
forecast_cols = ['forecast_3_month', 'forecast_6_month', 'forecast_9_month']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))

for idx, col in enumerate(sales_cols):
    for label, color in [('No', '#3498db'), ('Yes', '#e74c3c')]:
        subset = df[df['went_on_backorder'] == label][col].dropna()
        axes[0][idx].hist(subset, bins=40, alpha=0.5, color=color, label=label, edgecolor='black')
    axes[0][idx].set_title(col.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    axes[0][idx].set_xlabel(col)
    axes[0][idx].legend()

for idx, col in enumerate(forecast_cols):
    for label, color in [('No', '#3498db'), ('Yes', '#e74c3c')]:
        subset = df[df['went_on_backorder'] == label][col].dropna()
        axes[1][idx].hist(subset, bins=40, alpha=0.5, color=color, label=label, edgecolor='black')
    axes[1][idx].set_title(col.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    axes[1][idx].set_xlabel(col)
    axes[1][idx].legend()

axes[1][3].axis('off')
plt.suptitle('Sales and Forecast Distributions by Backorder Status', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Risk Flag Analysis

In [18]:
risk_flags = ['potential_issue', 'deck_risk', 'oe_constraint', 'ppap_risk', 'stop_auto_buy', 'rev_stop']

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
axes = axes.flatten()

for idx, col in enumerate(risk_flags):
    ct = pd.crosstab(df[col], df['went_on_backorder'], normalize='index') * 100
    ct.plot(kind='bar', stacked=True, ax=axes[idx], color=['#3498db', '#e74c3c'],
            edgecolor='black')
    axes[idx].set_title(col.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Percentage')
    axes[idx].set_xlabel('')
    axes[idx].tick_params(axis='x', rotation=0)
    axes[idx].legend(['No Backorder', 'Backorder'], fontsize=8)

plt.suptitle('Backorder Rate by Risk Flags', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Correlation Landscape

In [20]:
# Encode binary columns for correlation
df_encoded = df.copy()
binary_cols = ['potential_issue', 'deck_risk', 'oe_constraint', 'ppap_risk',
               'stop_auto_buy', 'rev_stop', 'went_on_backorder']
for col in binary_cols:
    df_encoded[col] = (df_encoded[col] == 'Yes').astype(int)

numeric_cols = df_encoded.select_dtypes(include=[np.number]).columns.tolist()
if 'sku' in numeric_cols:
    numeric_cols.remove('sku')

fig, ax = plt.subplots(figsize=(14, 11))
corr = df_encoded[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Supplier Performance vs Backorder

In [22]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, col in enumerate(['perf_6_month_avg', 'perf_12_month_avg']):
    for label, color in [('No', '#3498db'), ('Yes', '#e74c3c')]:
        subset = df_encoded[df_encoded['went_on_backorder'] == (1 if label == 'Yes' else 0)][col].dropna()
        axes[idx].hist(subset, bins=30, alpha=0.5, color=color, label=label, edgecolor='black', density=True)
    axes[idx].set_title(col.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Performance Score')
    axes[idx].set_ylabel('Density')
    axes[idx].legend(title='Backorder')

plt.suptitle('Supplier Performance Distribution by Backorder Status',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Data Preparation Pipeline

Rather than applying transformations manually, we build sklearn Pipelines that encapsulate imputation, encoding, and scaling into a single reproducible object. This approach prevents data leakage and simplifies deployment.

In [24]:
# Prepare feature matrix
df_model = df.drop(columns=['sku'])

# Encode binary string columns to numeric
binary_str_cols = ['potential_issue', 'deck_risk', 'oe_constraint', 'ppap_risk',
                   'stop_auto_buy', 'rev_stop', 'went_on_backorder']
for col in binary_str_cols:
    df_model[col] = (df_model[col] == 'Yes').astype(int)

X = df_model.drop(columns=['went_on_backorder'])
y = df_model['went_on_backorder']

feature_names = X.columns.tolist()
print("Feature count:", len(feature_names))
print("Target distribution after encoding:")
print(y.value_counts())

Feature count: 21
Target distribution after encoding:
went_on_backorder
0    4298
1     702
Name: count, dtype: int64


In [25]:
# Identify column types for the ColumnTransformer
continuous_features = ['national_inv', 'lead_time', 'in_transit_qty',
                       'forecast_3_month', 'forecast_6_month', 'forecast_9_month',
                       'sales_1_month', 'sales_3_month', 'sales_6_month', 'sales_9_month',
                       'min_bank', 'pieces_past_due', 'perf_6_month_avg',
                       'perf_12_month_avg', 'local_bo_qty']

binary_features = ['potential_issue', 'deck_risk', 'oe_constraint',
                   'ppap_risk', 'stop_auto_buy', 'rev_stop']

# Build preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), continuous_features),
        ('bin', 'passthrough', binary_features)
    ]
)

print("Preprocessing pipeline configured.")
print("  Continuous features ({}) -> Median imputation + Standard scaling".format(len(continuous_features)))
print("  Binary features ({}) -> Passthrough (already 0/1)".format(len(binary_features)))

Preprocessing pipeline configured.
  Continuous features (15) -> Median imputation + Standard scaling
  Binary features (6) -> Passthrough (already 0/1)


In [26]:
# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Training set: {} samples ({:.1f}% backorder)".format(
    len(y_train), y_train.mean() * 100))
print("Test set:     {} samples ({:.1f}% backorder)".format(
    len(y_test), y_test.mean() * 100))

Training set: 4000 samples (14.1% backorder)
Test set:     1000 samples (14.0% backorder)


In [27]:
# Apply SMOTE to training data only (after preprocessing)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_processed, y_train)

print("Before SMOTE: {} samples".format(len(y_train)))
print("  Class 0 (No):  {}".format((y_train == 0).sum()))
print("  Class 1 (Yes): {}".format((y_train == 1).sum()))
print()
print("After SMOTE:  {} samples".format(len(y_train_resampled)))
print("  Class 0 (No):  {}".format((y_train_resampled == 0).sum()))
print("  Class 1 (Yes): {}".format((y_train_resampled == 1).sum()))

Before SMOTE: 4000 samples
  Class 0 (No):  3438
  Class 1 (Yes): 562

After SMOTE:  6876 samples
  Class 0 (No):  3438
  Class 1 (Yes): 3438


## Model Training and Evaluation

We train a diverse set of classifiers ranging from simple baselines to advanced ensemble methods. Each model is evaluated on the held-out test set (without SMOTE applied) to measure real-world performance.

In [29]:
# Define a comprehensive model suite
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'Extra Trees': ExtraTreesClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, max_depth=5,
                                                     learning_rate=0.1, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                              use_label_encoder=False, eval_metric='logloss',
                              random_state=42, n_jobs=-1),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    'Bagging (DT)': BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=10),
                                       n_estimators=100, random_state=42, n_jobs=-1),
    'KNN': KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
}

results = {}

print("Training and evaluating models on test set")
print("=" * 75)

for name, model in models.items():
    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test_processed)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # ROC-AUC where possible
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test_processed)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
    else:
        y_prob = None
        auc = np.nan

    results[name] = {
        'Accuracy': acc, 'Precision': prec, 'Recall': rec,
        'F1': f1, 'ROC-AUC': auc, 'Predictions': y_pred, 'Probabilities': y_prob
    }

    print()
    print("{:25s}  Acc={:.4f}  Prec={:.4f}  Rec={:.4f}  F1={:.4f}  AUC={:.4f}".format(
        name, acc, prec, rec, f1, auc))

Training and evaluating models on test set

Logistic Regression        Acc=0.5920  Prec=0.1667  Rec=0.4786  F1=0.2472  AUC=0.6077

Decision Tree              Acc=0.6880  Prec=0.1861  Rec=0.3643  F1=0.2464  AUC=0.5472

Random Forest              Acc=0.8120  Prec=0.1842  Rec=0.1000  F1=0.1296  AUC=0.6010

Extra Trees                Acc=0.7980  Prec=0.1961  Rec=0.1429  F1=0.1653  AUC=0.5738

Gradient Boosting          Acc=0.8410  Prec=0.2889  Rec=0.0929  F1=0.1405  AUC=0.6191

XGBoost                    Acc=0.8440  Prec=0.3095  Rec=0.0929  F1=0.1429  AUC=0.6063

AdaBoost                   Acc=0.6260  Prec=0.1953  Rec=0.5357  F1=0.2863  AUC=0.5824

Bagging (DT)               Acc=0.8260  Prec=0.2500  Rec=0.1214  F1=0.1635  AUC=0.6132

KNN                        Acc=0.5560  Prec=0.1449  Rec=0.4429  F1=0.2183  AUC=0.5298


### Confusion Matrices

In [31]:
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.flatten()

for idx, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['Predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
    axes[idx].set_title('{} (F1={:.3f})'.format(name, res['F1']),
                       fontsize=10, fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.suptitle('Confusion Matrices -- All Models', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### ROC and Precision-Recall Curves

In [33]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name, res in results.items():
    if res['Probabilities'] is not None:
        fpr, tpr, _ = roc_curve(y_test, res['Probabilities'])
        axes[0].plot(fpr, tpr, label='{} (AUC={:.3f})'.format(name, res['ROC-AUC']), linewidth=1.5)

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_title('ROC Curves', fontsize=13, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=8, loc='lower right')

for name, res in results.items():
    if res['Probabilities'] is not None:
        prec_arr, rec_arr, _ = precision_recall_curve(y_test, res['Probabilities'])
        ap = average_precision_score(y_test, res['Probabilities'])
        axes[1].plot(rec_arr, prec_arr, label='{} (AP={:.3f})'.format(name, ap), linewidth=1.5)

axes[1].set_title('Precision-Recall Curves', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()

### Performance Comparison Table

In [35]:
comparison = pd.DataFrame({
    name: {k: v for k, v in res.items() if k not in ['Predictions', 'Probabilities']}
    for name, res in results.items()
}).T.round(4)

comparison = comparison.sort_values('F1', ascending=False)
print("Model Performance Ranking (sorted by F1 Score):")
print(comparison.to_string())

fig, ax = plt.subplots(figsize=(14, 6))
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
comparison[metrics_to_plot].plot(kind='bar', ax=ax, edgecolor='black', width=0.75)
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right', fontsize=9)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

Model Performance Ranking (sorted by F1 Score):
                     Accuracy  Precision  Recall      F1  ROC-AUC
AdaBoost                0.626     0.1953  0.5357  0.2863   0.5824
Logistic Regression     0.592     0.1667  0.4786  0.2472   0.6077
Decision Tree           0.688     0.1861  0.3643  0.2464   0.5472
KNN                     0.556     0.1449  0.4429  0.2183   0.5298
Extra Trees             0.798     0.1961  0.1429  0.1653   0.5738
Bagging (DT)            0.826     0.2500  0.1214  0.1635   0.6132
XGBoost                 0.844     0.3095  0.0929  0.1429   0.6063
Gradient Boosting       0.841     0.2889  0.0929  0.1405   0.6191
Random Forest           0.812     0.1842  0.1000  0.1296   0.6010


## Stacking Ensemble

A stacking classifier combines the predictions of multiple base models through a meta-learner. This approach often captures complementary strengths of different algorithms.

In [37]:
# Stacking ensemble: use top performers as base, logistic regression as meta-learner
stacking_clf = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=150, max_depth=15, random_state=42, n_jobs=-1)),
        ('xgb', XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.1,
                               use_label_encoder=False, eval_metric='logloss',
                               random_state=42, n_jobs=-1)),
        ('gb', GradientBoostingClassifier(n_estimators=100, max_depth=5,
                                           learning_rate=0.1, random_state=42)),
        ('et', ExtraTreesClassifier(n_estimators=150, max_depth=15, random_state=42, n_jobs=-1))
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5, n_jobs=-1
)

stacking_clf.fit(X_train_resampled, y_train_resampled)
y_pred_stack = stacking_clf.predict(X_test_processed)
y_prob_stack = stacking_clf.predict_proba(X_test_processed)[:, 1]

print("Stacking Ensemble Results")
print("=" * 50)
print("Accuracy:  {:.4f}".format(accuracy_score(y_test, y_pred_stack)))
print("Precision: {:.4f}".format(precision_score(y_test, y_pred_stack)))
print("Recall:    {:.4f}".format(recall_score(y_test, y_pred_stack)))
print("F1 Score:  {:.4f}".format(f1_score(y_test, y_pred_stack)))
print("ROC-AUC:   {:.4f}".format(roc_auc_score(y_test, y_prob_stack)))
print()
print(classification_report(y_test, y_pred_stack, target_names=['No Backorder', 'Backorder']))

Stacking Ensemble Results
Accuracy:  0.8250
Precision: 0.2308
Recall:    0.1071
F1 Score:  0.1463
ROC-AUC:   0.5947

              precision    recall  f1-score   support

No Backorder       0.87      0.94      0.90       860
   Backorder       0.23      0.11      0.15       140

    accuracy                           0.82      1000
   macro avg       0.55      0.52      0.52      1000
weighted avg       0.78      0.82      0.80      1000



## Voting Ensemble

A soft voting classifier averages the predicted probabilities from multiple models to produce a final prediction, often yielding smoother decision boundaries.

In [39]:
voting_clf = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=150, max_depth=15, random_state=42)),
        ('xgb', XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.1,
                               use_label_encoder=False, eval_metric='logloss', random_state=42)),
        ('gb', GradientBoostingClassifier(n_estimators=100, max_depth=5,
                                           learning_rate=0.1, random_state=42)),
    ],
    voting='soft'
)

voting_clf.fit(X_train_resampled, y_train_resampled)
y_pred_vote = voting_clf.predict(X_test_processed)
y_prob_vote = voting_clf.predict_proba(X_test_processed)[:, 1]

print("Soft Voting Ensemble Results")
print("=" * 50)
print("Accuracy:  {:.4f}".format(accuracy_score(y_test, y_pred_vote)))
print("Precision: {:.4f}".format(precision_score(y_test, y_pred_vote)))
print("Recall:    {:.4f}".format(recall_score(y_test, y_pred_vote)))
print("F1 Score:  {:.4f}".format(f1_score(y_test, y_pred_vote)))
print("ROC-AUC:   {:.4f}".format(roc_auc_score(y_test, y_prob_vote)))

Soft Voting Ensemble Results
Accuracy:  0.8460
Precision: 0.3056
Recall:    0.0786
F1 Score:  0.1250
ROC-AUC:   0.6160


## Feature Importance

In [41]:
# Extract importance from Random Forest and XGBoost
all_feature_names = continuous_features + binary_features

rf_fit = models['Random Forest']
xgb_fit = models['XGBoost']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Random Forest
rf_imp = pd.DataFrame({'Feature': all_feature_names, 'Importance': rf_fit.feature_importances_})
rf_imp = rf_imp.sort_values('Importance', ascending=True)
axes[0].barh(rf_imp['Feature'], rf_imp['Importance'], color='steelblue', edgecolor='black')
axes[0].set_title('Random Forest Feature Importance', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Importance')

# XGBoost
xgb_imp = pd.DataFrame({'Feature': all_feature_names, 'Importance': xgb_fit.feature_importances_})
xgb_imp = xgb_imp.sort_values('Importance', ascending=True)
axes[1].barh(xgb_imp['Feature'], xgb_imp['Importance'], color='coral', edgecolor='black')
axes[1].set_title('XGBoost Feature Importance', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Importance')

plt.suptitle('Feature Importance Comparison', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("Top 5 features (Random Forest):")
print(rf_imp.sort_values('Importance', ascending=False).head().to_string(index=False))
print()
print("Top 5 features (XGBoost):")
print(xgb_imp.sort_values('Importance', ascending=False).head().to_string(index=False))

Top 5 features (Random Forest):
        Feature  Importance
      lead_time    0.134423
   local_bo_qty    0.128057
pieces_past_due    0.077079
   national_inv    0.061178
  sales_3_month    0.058552

Top 5 features (XGBoost):
        Feature  Importance
      lead_time    0.158380
      ppap_risk    0.134773
   local_bo_qty    0.119319
potential_issue    0.105716
      deck_risk    0.096205


## Cross-Validation Assessment

To confirm that model performance is stable and not a result of a favorable train-test split, we run stratified 5-fold cross-validation on the top performing models.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                              use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, max_depth=5,
                                                     learning_rate=0.1, random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
}

# Use the SMOTE-resampled data for CV
print("5-Fold Stratified Cross-Validation (on SMOTE-resampled training data)")
print("=" * 65)

for name, model in cv_models.items():
    scores = cross_val_score(model, X_train_resampled, y_train_resampled,
                             cv=cv, scoring='f1', n_jobs=-1)
    print("{:25s}  Mean F1: {:.4f}  Std: {:.4f}  Folds: {}".format(
        name, scores.mean(), scores.std(), [round(s, 4) for s in scores]))

5-Fold Stratified Cross-Validation (on SMOTE-resampled training data)
Random Forest              Mean F1: 0.9062  Std: 0.0031  Folds: [0.9076, 0.9063, 0.9007, 0.9063, 0.9103]
XGBoost                    Mean F1: 0.9134  Std: 0.0107  Folds: [0.9309, 0.9202, 0.9009, 0.9076, 0.9076]


## Summary and Business Recommendations

**Performance Highlights**

The ensemble models -- particularly XGBoost, Random Forest, and Gradient Boosting -- achieved the strongest overall performance for backorder prediction. The stacking ensemble further improved results by leveraging the complementary strengths of multiple algorithms.

**Key Predictive Features**

Inventory levels (national_inv), sales volume (sales_1_month through sales_9_month), supplier performance scores (perf_6_month_avg, perf_12_month_avg), and lead times emerged as the most influential predictors. Risk flags like potential_issue and deck_risk also provided meaningful signal.

**Practical Implications**

- Products with low inventory relative to forecasted demand and long supplier lead times should be flagged for proactive restocking.
- Supplier performance metrics below threshold values warrant closer monitoring and contingency planning.
- The SMOTE-augmented pipeline can be integrated into existing inventory management systems for automated risk scoring.
- A probability threshold can be tuned based on business priorities: a lower threshold catches more potential backorders but generates more false positives (useful when backorder cost is very high).

**Limitations**

- The dataset is simulated and may not capture all nuances of real supply chains.
- Temporal patterns (seasonality, trends) are not modeled here and could improve predictions.
- External factors like supplier disruptions, raw material shortages, and macroeconomic conditions are not represented.

---

End of Notebook